In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableMap

In [8]:
# Load the textfile
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["."],
    chunk_size = 300,chunk_overlap = 50
)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain and CrewAI represent two complementary paradigms for building applications with large \nlanguage models, and understanding how they fit together is essential for anyone designing \nmodern AI agent systems; LangChain, at its core, is a framework for composing chains of \noperations around LLMs, providing abstractions for models, prompts, memory, retrievers, tools, \nand output parsers that let developers wire together deterministic pipelines where each step \ntransforms data before passing it to the next, and its ecosystem includes LangGraph for stateful, \ngraph-based orchestration, LangSmith for tracing and observability, and LangServe for deploying chains \nas APIs, making it ideal for scenarios where you need fine-grained control over the flow of data, \nretrieval-augmented generation, document processing, conversational memory, and integration with dozens \nof vector stores, APIs, and data source

In [9]:
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstores = FAISS.from_documents(docs,embedding_model)
retriever = vectorstores.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3}
)
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fef502323c0>, search_type='mmr', search_kwargs={'k': 3})

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatGroq(
     model="openai/gpt-oss-20b",
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )

In [11]:
query_decomposer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query decomposer. Break the user's complex query into "
     "clear, self-contained sub-queries that can be answered independently. "
     "Preserve intent and entities. Do not answer the query. "
     "Return ONLY a JSON list of strings, e.g. [\"sub-query 1\", \"sub-query 2\"]."),
    ("human", "Query: {query}")
])
decomposition_chain = query_decomposer_prompt | llm | StrOutputParser()

In [14]:
query = "How does langchain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"query":query})
decomposition_question

'["What memory mechanisms does LangChain provide?","How does LangChain implement agents?","What memory mechanisms does CrewAI provide?","How does CrewAI implement agents?","How does LangChain\'s use of memory and agents differ from CrewAI\'s?"]'

In [15]:
prompt = PromptTemplate.from_template("""
    Answer the question based on the context below.
    
    context:{context}
    Question:{input}"""
)
document_chain = create_stuff_documents_chain(llm,prompt)


In [16]:
# Full rag pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"query" : user_query})
    sub_questions = [q.strip("-.1234567890").strip() for q in sub_qs_text.split("\n") if q.strip()]

    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = document_chain.invoke({"input":subq, "context":docs})
        results.append(f"Q: {subq} \nA: {result}")
    return "\n\n".join(results)


In [17]:
query = "How does langchain use memory and agents compared to crewai?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("Final answer:\n")
print(final_answer)

Final answer:

Q: ["How does LangChain implement memory management compared to CrewAI?", "How does LangChain implement agents compared to CrewAI?"] 
A: **1. Memory Management**

| Aspect | LangChain | CrewAI |
|--------|-----------|--------|
| **Primary focus** | Explicit, fine‑grained memory abstractions (e.g., conversation memory, vector‑store memory, retrieval‑augmented memory). | Implicit context sharing between agents; memory is usually handled by the underlying LLM or by the tools the agents call. |
| **How it’s implemented** | • `Memory` classes (e.g., `ConversationBufferMemory`, `VectorStoreRetrieverMemory`) that store past turns or retrieved documents in a backing store (in‑memory, Redis, Pinecone, etc.).  <br>• Memory is passed as a component of a chain, so every step can read/write to it.  <br>• Supports “retrieval‑augmented generation” by automatically querying a vector store before a prompt. | • Agents receive a “context” string that is passed along the workflow.  <br>• No